In [0]:
from pyspark.sql.functions import *

### Establishing the connection with SILVER

In [0]:
spark.conf.set("fs.azure.account.key.sa1storage.dfs.core.windows.net", "G1RfRXHyK97KaTGSJXAD1RqeXkcX0WRYxtUi4g/pUZx7GxFi3MWAEvU7uU9yh7I27Syh/IUORuea+AStsT8Lzw==")

### Loading all the DIMENSIONS

In [0]:
# Customer Dimension

df_cust = spark.sql(
    """
    SELECT *
    FROM delta.`abfss://gold@sa1storage.dfs.core.windows.net/Dim_Customer`
    """
)

df_cust.limit(20).display()

Cust_ID,Cust_Name,Email,Country,Dim_Cust_Key
1935,Beth Hernandez,aliciamarks@gmail.com,India,1
1181,James Hurley,skinnerrussell@lowery-chaney.com,Iraq,2
1248,Nicholas Jensen,garciaangela@levine.net,New Zealand,3
1410,Amanda Lopez,donaldmendoza@richard-anderson.com,Namibia,4
1989,Andre Lin,jacobsmith@skinner-chapman.com,Nepal,5
1349,Alex Stone,jordantaylor@johnson.com,Equatorial Guinea,6
1669,Diana Eaton,kaylathompson@miranda.com,Georgia,7
1944,Kyle Gray,jesus71@ortiz.com,Myanmar,8
1257,Kurt Roberts,hansenanthony@vargas.net,Trinidad and Tobago,9
1840,Aaron Hall,bethwilson@miles.com,Turkey,10


In [0]:
# Product Dimension

df_prod = spark.sql(
    """
    SELECT *
    FROM delta.`abfss://gold@sa1storage.dfs.core.windows.net/Dim_Product`
    """
)

df_prod.limit(20).display()

Prod_ID,Prod_Cat,Prod_Name,Prod_Price,Dim_Prod_Key
537,Sports,Phone,342.54,1
565,Sports,Tennis Racket,758.11,2
535,Furniture,Bicycle,870.46,3
508,Beauty,T-Shirt,86.85,4
551,Electronics,Perfume,108.43,5
580,Beauty,Table,384.50,6
532,Sports,Shoes,781.38,7
507,Furniture,Bicycle,782.90,8
565,Beauty,Table,787.37,9
577,Sports,Shoes,368.88,10


In [0]:
# Date Dimension

df_date = spark.sql(
    """
    SELECT *
    FROM delta.`abfss://gold@sa1storage.dfs.core.windows.net/Dim_Date`
    """
)

df_date.limit(20).display()

Order_Date,Year,Month,Day,Dim_Date_Key
2025-02-16,2025,2,16,1
2024-09-18,2024,9,18,2
2023-07-15,2023,7,15,3
2023-11-08,2023,11,8,4
2024-08-27,2024,8,27,5
2024-06-04,2024,6,4,6
2024-05-30,2024,5,30,7
2023-09-14,2023,9,14,8
2024-10-24,2024,10,24,9
2023-09-19,2023,9,19,10


In [0]:
# Region Dimension

df_region = spark.sql(
    """
    SELECT *
    FROM delta.`abfss://gold@sa1storage.dfs.core.windows.net/Dim_Region`
    """
)

df_region.limit(20).display()

SalesRegion,Dim_Region_Key
South,1
Central,2
East,3
West,4
North,5


### Loading the SILVER Data

In [0]:
df_silver = spark.sql(
    """
    SELECT *
    FROM parquet.`abfss://silver@sa1storage.dfs.core.windows.net/cleanseddata`
    """
)

df_silver.limit(20).display()

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,Country,ProductID,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,SalesRegion
38,2024-04-10,1696,George Wright,ibarnes@davis.com,Guatemala,579,Sports,Phone,5,583.72,2918.60,Central
433,2025-02-06,1121,Rachael Paul,rachael01@richmond.com,Bahrain,536,Furniture,Laptop,8,178.08,1424.64,Central
464,2024-09-24,1939,Matthew Lane,perryjoseph@gmail.com,Czech Republic,538,Clothing,Laptop,3,891.24,2673.72,West
618,2025-02-09,1354,Douglas Park,erowland@davis-doyle.info,Bulgaria,573,Furniture,Shoes,5,182.88,914.40,Central
632,2023-10-28,1187,Kelly Fisher,iparker@yahoo.com,Panama,539,Furniture,Shoes,9,782.59,7043.31,West
885,2023-10-09,1990,Gary Ray,christopherwilson@gmail.com,Saint Martin,552,Clothing,Laptop,1,842.54,842.54,East
969,2023-08-22,1249,Christine Curtis,thomas75@yahoo.com,Northern Mariana Islands,598,Electronics,Perfume,4,145.78,583.12,South
6,2025-06-12,1525,Elizabeth Cunningham,karen47@lee-cox.com,Azerbaijan,537,Clothing,Shoes,5,978.02,4890.10,West
112,2023-10-02,1442,Michelle Craig,christianguerrero@hotmail.com,Nepal,553,Beauty,Table,9,84.37,759.33,North
311,2024-11-21,1196,Rhonda Hardy,alexis46@hotmail.com,Mongolia,545,Electronics,Bicycle,4,13.15,52.60,West


### Creating FACT Table

In [0]:
df_fact = df_silver.join(df_cust, df_silver['CustomerID'] == df_cust['Cust_ID'], 'left').join(df_prod, df_silver['ProductID'] == df_prod['Prod_ID'], 'left').join(df_date, df_silver['OrderDate'] == df_date['Order_Date'], 'left').join(df_region, df_silver['SalesRegion'] == df_region['SalesRegion'], 'left').select(df_cust['Dim_Cust_Key'], df_prod['Dim_Prod_Key'], df_date['Dim_Date_Key'], df_region['Dim_Region_Key'], df_silver['Quantity'], df_silver['UnitPrice'], df_silver['TotalPrice'])

df_fact.limit(20).display()

Dim_Cust_Key,Dim_Prod_Key,Dim_Date_Key,Dim_Region_Key,Quantity,UnitPrice,TotalPrice
553,952,269,2,5,583.72,2918.60
553,714,269,2,5,583.72,2918.60
553,654,269,2,5,583.72,2918.60
553,653,269,2,5,583.72,2918.60
553,465,269,2,5,583.72,2918.60
553,180,269,2,5,583.72,2918.60
553,76,269,2,5,583.72,2918.60
543,952,269,2,5,583.72,2918.60
543,714,269,2,5,583.72,2918.60
543,654,269,2,5,583.72,2918.60


#### UP-SERT

In [0]:
from delta.tables import DeltaTable

table_name = 'gold.Fact_Sales'

path = 'abfss://gold@sa1storage.dfs.core.windows.net/Fact_Sales'

if DeltaTable.isDeltaTable(spark, path):  
  deltaTable = DeltaTable.forPath(spark, path)
  deltaTable.alias('target').merge(df_fact.alias('source'), 'target.Dim_Cust_Key == source.Dim_Cust_Key and target.Dim_Prod_Key == source.Dim_Prod_Key and target.Dim_Date_Key == source.Dim_Date_Key and target.Dim_Region_Key == source.Dim_Region_Key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
  print('The TABLE exists')

else:
  df_fact.write.format('delta').mode('overwrite').option('mergeschema', 'true').save(path)
  print('The TABLE is created')

The TABLE exists
